In [ ]:
from pathlib import Path
import importlib.util
import os
import subprocess
import sys

REQUIRE_REAL_GDN = True

REQUIRED_PIP_PACKAGES = {
    "miditok": "miditok",
    "pretty_midi": "pretty_midi",
    "ncps": "ncps",
    "safetensors": "safetensors",
    "matplotlib": "matplotlib",
    "huggingface_hub": "huggingface_hub",
}

def ensure_runtime_dependencies() -> None:
    missing = [spec for name, spec in REQUIRED_PIP_PACKAGES.items() if importlib.util.find_spec(name) is None]
    if not missing:
        print("Runtime dependencies already satisfied.")
        return
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--disable-pip-version-check", *missing])

def can_write_to(path: Path) -> bool:
    try:
        path.mkdir(parents=True, exist_ok=True)
        probe = path / ".copilot_write_probe"
        probe.write_text("ok", encoding="utf-8")
        probe.unlink()
        return True
    except Exception:
        return False

def _candidate_roots() -> list[Path]:
    return [
        Path.cwd().resolve(),
        Path.cwd().resolve().parent,
        Path("/kaggle/working/piano_midi_model"),
        Path("/kaggle/input/magic_midi"),
        Path("/kaggle/input/magic_midi/piano_midi_model"),
        Path("/kaggle/input/magic-midi"),
        Path("/kaggle/input/magic-midi/piano_midi_model"),
        Path("/kaggle/input/datasets/chickaboomcmurtrie/magic_midi"),
        Path("/kaggle/input/datasets/chickaboomcmurtrie/magic_midi/piano_midi_model"),
        Path("/kaggle/input/datasets/chickaboomcmurtrie/magic-midi"),
        Path("/kaggle/input/datasets/chickaboomcmurtrie/magic-midi/piano_midi_model"),
    ]

def find_project_root() -> Path:
    anchor = Path("training/ablation_runner.py")
    for probe in _candidate_roots():
        if not probe.exists():
            continue
        if (probe / anchor).exists() and (probe / "training" / "__init__.py").exists():
            return probe
    raise FileNotFoundError(
        f"Could not locate project root containing {anchor}. "
        "Attach the repo dataset at one of the known Kaggle mount paths."
    )

def _ensure_flash_linear_attention() -> None:
    from model.blocks import gdn_block

    if gdn_block.GDN_AVAILABLE:
        print("Real GDN kernels already available.")
        return

    install_specs = ["flash-linear-attention", "flash-linear-attention==0.4.2"]
    last_error = None
    for spec in install_specs:
        try:
            subprocess.check_call([
                sys.executable,
                "-m",
                "pip",
                "install",
                "--quiet",
                "--disable-pip-version-check",
                spec,
            ])
            if gdn_block.try_import_fla():
                print(f"Installed {spec} and loaded real GDN kernels.")
                return
            print(f"Install succeeded for {spec} but GDN import is still unavailable.")
        except Exception as exc:
            last_error = exc
            print(f"{spec} install failed: {exc}")

    raise RuntimeError(
        "Real GDN kernels are unavailable. Install flash-linear-attention or use the fallback path."
    ) from last_error

ensure_runtime_dependencies()
PROJECT_ROOT = find_project_root()
OUTPUT_BASE = Path("/kaggle/working") if can_write_to(Path("/kaggle/working")) else PROJECT_ROOT
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.environ["PYTHONPATH"] = str(PROJECT_ROOT) + os.pathsep + os.environ.get("PYTHONPATH", "")

_ensure_flash_linear_attention()
from model.blocks import gdn_block

if REQUIRE_REAL_GDN and not gdn_block.GDN_AVAILABLE:
    raise RuntimeError("Real GDN kernels are unavailable. Notebook is strict and blocks fallback GDN.")

print(f"Project root: {PROJECT_ROOT}")
print(f"Output base: {OUTPUT_BASE}")
print(f"GDN_AVAILABLE: {gdn_block.GDN_AVAILABLE}")

In [ ]:
import numpy as np
import torch

MAX_PIECES = 100_000
EPOCHS = 20
SEED = 42

SEED_LENGTH = 512
CONTINUATION_LENGTH = 1536
MAX_SEQUENCE_LENGTH = SEED_LENGTH + CONTINUATION_LENGTH

TARGET_EFFECTIVE_BATCH = 32
BATCH_PER_GPU = 2
LEARNING_RATE = 2e-4
WARMUP_RATIO = 0.1
WARMUP_STEPS = 0
WARMUP_TARGET_EPOCH_FRACTION = 0.75
WARMUP_MAX_EPOCH_FRACTION = 1.0
MIN_LR_RATIO = 0.1
WEIGHT_DECAY = 0.01
LABEL_SMOOTHING = 0.1
MAX_GRAD_NORM = 1.0
LOG_EVERY_N_STEPS = 50
SAVE_EVERY_N_STEPS = 1000
SAVE_EVERY_N_EPOCHS = 5
KEEP_EVERY_N_EPOCHS = 10
MAX_CHECKPOINTS = 8

AUTO_RESUME = False
RESUME_FROM_CHECKPOINT = ""
RESUME_MODE = "remaining"

ALLOW_FALLBACK_GDN = False
RUN_DRY_RUN = False
RUN_RESUME_TRAINING = True
RUN_TRAINING_FROM_SCRATCH = False
RUN_TRAINING = False  # Deprecated in this notebook; kept for compatibility.
USE_DDP_FOR_MULTI_GPU = True

PRETOKENIZED_ROOT_CANDIDATES = [
    PROJECT_ROOT / "processed",
    Path("/kaggle/input/datasets/chickaboomcmurtrie/pulse88-tokenize-100k"),
    Path("/kaggle/input/datasets/chickaboomcmurtrie/pulse88-tokenize-100k/tokenized"),
    Path("/kaggle/input/godzilla-tokenized-100k/tokenized"),
    Path("/kaggle/input/godzilla-tokenized-100k"),
    Path("/kaggle/input/godzilla-piano-tokenized-100k/tokenized"),
    Path("/kaggle/input/godzilla-piano-tokenized-100k"),
    Path("/kaggle/input/godzilla-tokenized-15k/tokenized"),
    Path("/kaggle/input/godzilla-tokenized-15k"),
]

GDN_PROFILE_40M = {
    "d_model": 640,
    "n_layers": 13,
    "attention_every_n_layers": 2,
    "gdn_inner_ratio": 0.5,
    "gdn_num_heads": 4,
    "gqa_num_heads": 8,
    "gqa_groups": 4,
}

GPU_COUNT = torch.cuda.device_count() if torch.cuda.is_available() else 0
SINGLE_PROCESS_BATCH_SIZE = max(1, int(BATCH_PER_GPU))
GLOBAL_BATCH_BASE = max(1, int(BATCH_PER_GPU) * max(1, int(GPU_COUNT)))
GRAD_ACCUM_STEPS = max(1, int(np.ceil(float(TARGET_EFFECTIVE_BATCH) / float(GLOBAL_BATCH_BASE))))
NUM_WORKERS = max(2, int(max(1, GPU_COUNT)) * 2)
WILL_USE_DDP = bool(USE_DDP_FOR_MULTI_GPU and GPU_COUNT > 1 and REQUIRE_REAL_GDN)

run_tag = f"sub100m_e_40m_{MAX_PIECES // 1000}k"
OUTPUT_DIR = OUTPUT_BASE / "outputs" / run_tag
SOURCE_MANIFEST_DIR = OUTPUT_DIR / "source_pretokenized"
PRETOKENIZED_MANIFEST = SOURCE_MANIFEST_DIR / "manifest.json"
RESULT_JSON = OUTPUT_DIR / "variant_e_40m_result.json"

# Epoch bundles are copied to epoch_XXX folders after every epoch.
EPOCH_EXPORT_ROOT = OUTPUT_DIR / "kaggle_model_exports"

# Optional: command template executed after each epoch bundle is created.
# Leave blank to skip automatic upload.
# Available placeholders: {epoch}, {epoch_dir}, {output_dir}, {checkpoint_dir}
EPOCH_UPLOAD_CMD_TEMPLATE = ""

print(f"GPU count: {GPU_COUNT}")
print(f"Per-GPU batch: {BATCH_PER_GPU} | Global base batch: {GLOBAL_BATCH_BASE}")
print(f"Grad accumulation: {GRAD_ACCUM_STEPS} | Target effective batch: {TARGET_EFFECTIVE_BATCH}")
print(f"Step checkpoint cadence: every {int(SAVE_EVERY_N_STEPS):,} optimizer steps")
print(f"Warmup target/cap (epochs): {WARMUP_TARGET_EPOCH_FRACTION:.2f}/{WARMUP_MAX_EPOCH_FRACTION:.2f}")
print(f"Will use DDP path: {WILL_USE_DDP}")
print(f"Output dir: {OUTPUT_DIR}")
print(f"Epoch export root: {EPOCH_EXPORT_ROOT}")

In [ ]:
import json
import numpy as np

def first_existing_path(paths):
    for path in paths:
        if path.exists():
            return path
    return None

def load_manifest_count(manifest_path: Path) -> int:
    raw = json.loads(manifest_path.read_text(encoding="utf-8"))
    return len(raw) if isinstance(raw, list) else 0

def build_manifest_from_npz(npz_root: Path, manifest_path: Path) -> int:
    npz_files = sorted(npz_root.rglob("*.npz"))
    if not npz_files:
        raise FileNotFoundError(f"No .npz files found under: {npz_root}")
    manifest = []
    for npz_path in npz_files:
        try:
            with np.load(npz_path, allow_pickle=False) as pack:
                if "tokens" not in pack.files:
                    continue
                token_len = int(np.asarray(pack["tokens"]).shape[0])
        except Exception:
            continue
        manifest.append({"md5": npz_path.stem, "npz_path": str(npz_path), "length": token_len, "source_path": str(npz_path.parent)})
    manifest_path.parent.mkdir(parents=True, exist_ok=True)
    manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    return len(manifest)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PRETOKENIZED_ROOT = first_existing_path(PRETOKENIZED_ROOT_CANDIDATES)
if PRETOKENIZED_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the pulse88-tokenize-100k dataset root under /kaggle/input."
    )

PRETOKENIZED_MANIFEST = first_existing_path([
    PRETOKENIZED_ROOT / "manifest.json",
    PRETOKENIZED_ROOT / "processed_pretokenized" / "manifest.json",
    PRETOKENIZED_ROOT / "tokenized" / "manifest.json",
    PRETOKENIZED_MANIFEST,
])

if PRETOKENIZED_MANIFEST is None:
    PRETOKENIZED_MANIFEST = SOURCE_MANIFEST_DIR / "manifest.json"
    print("No prebuilt manifest found in the mounted dataset; building a raw source manifest in /kaggle/working.")
    count = build_manifest_from_npz(PRETOKENIZED_ROOT, PRETOKENIZED_MANIFEST)
else:
    count = load_manifest_count(PRETOKENIZED_MANIFEST)

print(f"Pretokenized root: {PRETOKENIZED_ROOT}")
print(f"Manifest path: {PRETOKENIZED_MANIFEST}")
print(f"Manifest entries: {count:,}")
if count < MAX_PIECES:
    print("WARNING: fewer than 100k entries available; run will use all available pieces.")

def estimate_steps_per_epoch(entry_count: int) -> int:
    total_entries = max(2, int(entry_count))
    n_val = max(1, int(round(float(total_entries) * 0.1)))
    n_val = min(n_val, total_entries - 1) if total_entries > 1 else 1
    n_train = max(1, total_entries - n_val)

    if bool(WILL_USE_DDP):
        per_rank_train = int(np.ceil(float(n_train) / float(max(1, GPU_COUNT))))
        micro_batches = int(np.ceil(float(per_rank_train) / float(max(1, BATCH_PER_GPU))))
    else:
        micro_batches = int(np.ceil(float(n_train) / float(max(1, SINGLE_PROCESS_BATCH_SIZE))))

    return max(1, int(np.ceil(float(micro_batches) / float(max(1, GRAD_ACCUM_STEPS)))))

ESTIMATED_STEPS_PER_EPOCH = estimate_steps_per_epoch(count)
ESTIMATED_TOTAL_STEPS = int(max(1, ESTIMATED_STEPS_PER_EPOCH) * max(1, int(EPOCHS)))

max_warmup_steps = max(
    1,
    int(round(float(WARMUP_MAX_EPOCH_FRACTION) * float(ESTIMATED_STEPS_PER_EPOCH))),
)
target_warmup_steps = max(
    1,
    int(round(float(WARMUP_TARGET_EPOCH_FRACTION) * float(ESTIMATED_STEPS_PER_EPOCH))),
)

if int(WARMUP_STEPS) > 0:
    resolved_warmup_steps = min(int(WARMUP_STEPS), int(max_warmup_steps))
else:
    ratio_warmup_steps = max(
        1,
        int(round(float(WARMUP_RATIO) * float(ESTIMATED_TOTAL_STEPS))),
    )
    resolved_warmup_steps = min(
        max(1, min(int(target_warmup_steps), int(ratio_warmup_steps))),
        int(max_warmup_steps),
    )

WARMUP_STEPS = int(resolved_warmup_steps)
WARMUP_RATIO = float(WARMUP_STEPS) / float(max(1, ESTIMATED_TOTAL_STEPS))

print(f"Estimated optimizer steps/epoch: {ESTIMATED_STEPS_PER_EPOCH:,}")
print(f"Resolved warmup steps: {WARMUP_STEPS:,} ({WARMUP_STEPS / max(1, ESTIMATED_STEPS_PER_EPOCH):.2f} epoch)")
print(f"Resolved warmup ratio over total training: {WARMUP_RATIO:.6f}")

In [ ]:
import torch

from data.tokenizer import CustomDeltaTokenizer
from model.variant_e import VariantEConfig, VariantEModel
from training.ablation_runner import _variant_backend_status

def resolve_divisible_heads(width: int, requested_heads: int) -> int:
    heads = max(1, min(int(requested_heads), int(width)))
    while heads > 1 and (int(width) % heads) != 0:
        heads -= 1
    return max(1, heads)

tokenizer = CustomDeltaTokenizer(include_special_tokens=False)
event_size = int(getattr(tokenizer, "event_size", 1))
if event_size != 4:
    raise RuntimeError(f"Expected tokenizer event_size=4, got {event_size}")
if (SEED_LENGTH % event_size) != 0 or (CONTINUATION_LENGTH % event_size) != 0:
    raise RuntimeError("Seed and continuation lengths must be divisible by event_size.")

d_model = int(GDN_PROFILE_40M["d_model"])
gdn_inner_dim = max(128, int(round(float(d_model) * float(GDN_PROFILE_40M["gdn_inner_ratio"]))))
gdn_heads = resolve_divisible_heads(gdn_inner_dim, int(GDN_PROFILE_40M["gdn_num_heads"]))
gqa_heads = resolve_divisible_heads(d_model, int(GDN_PROFILE_40M["gqa_num_heads"]))
gqa_groups = max(1, min(int(GDN_PROFILE_40M["gqa_groups"]), int(gqa_heads)))
while gqa_groups > 1 and (gqa_heads % gqa_groups) != 0:
    gqa_groups -= 1

model = VariantEModel(VariantEConfig(vocab_size=int(tokenizer.vocab_size), d_model=d_model, n_layers=int(GDN_PROFILE_40M["n_layers"]), max_sequence_length=int(MAX_SEQUENCE_LENGTH), gdn_inner_dim=int(gdn_inner_dim), gdn_num_heads=int(gdn_heads), gqa_num_heads=int(gqa_heads), gqa_groups=int(gqa_groups), attention_every_n_layers=int(GDN_PROFILE_40M["attention_every_n_layers"])))
params = int(sum(p.numel() for p in model.parameters()))
if not (38_000_000 <= params <= 42_000_000):
    raise RuntimeError(f"Expected ~40M params, got {params:,}")

backend_status = _variant_backend_status(model)
if REQUIRE_REAL_GDN and bool(backend_status.get("gdn_using_fallback", False)):
    raise RuntimeError("Variant E instantiated with fallback GDN. Real GDN is required.")

attention_layers = [i for i, block in enumerate(model.layers) if bool(getattr(block, "use_attention", False))]
if not attention_layers:
    raise RuntimeError("No sparse attention anchor layers detected.")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()
probe_seq = min(256, int(MAX_SEQUENCE_LENGTH))
probe_tokens = torch.randint(0, int(tokenizer.vocab_size), (1, probe_seq), dtype=torch.long, device=device)
probe_onsets = torch.arange(probe_seq, dtype=torch.float32, device=device).unsqueeze(0) * 0.1
with torch.no_grad():
    logits, _ = model(token_ids=probe_tokens, onset_times=probe_onsets, memory=None, return_memory=True, position_offset=0)
if tuple(logits.shape) != (1, probe_seq, int(tokenizer.vocab_size)):
    raise RuntimeError(f"Unexpected logits shape: {tuple(logits.shape)}")

print("Architecture preflight passed.")
print(f"Params: {params:,} ({params / 1e6:.2f}M)")
print(f"Backend status: {backend_status}")
print(f"Attention anchor layers: {attention_layers}")

In [ ]:
from pathlib import Path
import os

KAGGLE_MODEL_LABEL = "Variant E 40M GDN Piano"

CHECKPOINT_DATASET_CANDIDATES = [
    Path("/kaggle/input/models/chickaboomcmurtrie/variant-e-40m-gdn-piano/pytorch/default/1"),
    Path("/kaggle/input/variant-e-40m-gdn-piano"),
    Path("/kaggle/input/Variant-E-40M-GDN-Piano"),
    Path("/kaggle/input/variant_e_40m_gdn_piano"),
    Path("/kaggle/input/datasets/chickaboomcmurtrie/variant-e-40m-gdn-piano"),
    Path("/kaggle/input/datasets/chickaboomcmurtrie/Variant-E-40M-GDN-Piano"),
    Path("/kaggle/input/datasets/chickaboomcmurtrie/variant_e_40m_gdn_piano"),
    OUTPUT_DIR / "checkpoints",
]

CHECKPOINT_FILE_PRIORITY = [
    "latest_state.pt",
    "latest.safetensors",
    "best_state.pt",
    "best.safetensors",
]

def resolve_resume_checkpoint_from_sources() -> Path | None:
    raw = str(RESUME_FROM_CHECKPOINT).strip()
    if raw:
        manual = Path(raw).expanduser()
        if manual.exists() and manual.is_file():
            return manual.resolve()
        if manual.exists() and manual.is_dir():
            for name in CHECKPOINT_FILE_PRIORITY:
                candidates = sorted(manual.rglob(name), key=lambda p: p.stat().st_mtime, reverse=True)
                if candidates:
                    return candidates[0].resolve()
        print(f"WARNING: RESUME_FROM_CHECKPOINT was set but not found: {manual}")

    existing_roots = [path for path in CHECKPOINT_DATASET_CANDIDATES if path.exists()]
    ranked: list[tuple[int, float, Path]] = []
    for root in existing_roots:
        for priority, filename in enumerate(CHECKPOINT_FILE_PRIORITY):
            matches = sorted(root.rglob(filename), key=lambda p: p.stat().st_mtime, reverse=True)
            if matches:
                ranked.append((int(priority), -float(matches[0].stat().st_mtime), matches[0]))

    if not ranked:
        return None

    ranked.sort(key=lambda item: (item[0], item[1]))
    return ranked[0][2].resolve()

resolved_ckpt = resolve_resume_checkpoint_from_sources()
if resolved_ckpt is not None:
    RESUME_FROM_CHECKPOINT = str(resolved_ckpt)
    AUTO_RESUME = True
    RESUME_MODE = "remaining"
    RUN_RESUME_TRAINING = True
    RUN_TRAINING_FROM_SCRATCH = False
    RUN_TRAINING = False
    print(f"Resume checkpoint detected ({KAGGLE_MODEL_LABEL}): {RESUME_FROM_CHECKPOINT}")
else:
    RUN_RESUME_TRAINING = False
    print("No resume checkpoint found in model/input roots. Set RESUME_FROM_CHECKPOINT manually.")

if not str(EPOCH_UPLOAD_CMD_TEMPLATE).strip():
    EPOCH_UPLOAD_CMD_TEMPLATE = str(os.environ.get("EPOCH_UPLOAD_CMD_TEMPLATE", "")).strip()

EPOCH_EXPORT_ROOT.mkdir(parents=True, exist_ok=True)

print(f"AUTO_RESUME={AUTO_RESUME} | RESUME_MODE={RESUME_MODE}")
print(f"RUN_RESUME_TRAINING={RUN_RESUME_TRAINING} | RUN_TRAINING_FROM_SCRATCH={RUN_TRAINING_FROM_SCRATCH}")
if str(RESUME_FROM_CHECKPOINT).strip():
    print(f"RESUME_FROM_CHECKPOINT={RESUME_FROM_CHECKPOINT}")
print(f"EPOCH_EXPORT_ROOT={EPOCH_EXPORT_ROOT}")
print(f"EPOCH_UPLOAD_CMD_TEMPLATE set: {bool(str(EPOCH_UPLOAD_CMD_TEMPLATE).strip())}")

In [ ]:
from dataclasses import replace
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

import numpy as np

from training.sub100m_unified import UnifiedSub100mConfig, run_unified_sub100m

cfg = UnifiedSub100mConfig(
    variant="e",
    output_dir=str(OUTPUT_DIR),
    pretokenized_manifest=str(PRETOKENIZED_MANIFEST),
    pretokenized_root=str(PRETOKENIZED_ROOT),
    max_pieces=int(MAX_PIECES),
    seed=int(SEED),
    seed_length=int(SEED_LENGTH),
    continuation_length=int(CONTINUATION_LENGTH),
    max_sequence_length=int(MAX_SEQUENCE_LENGTH),
    epochs=int(EPOCHS),
    batch_size=int(SINGLE_PROCESS_BATCH_SIZE),
    grad_accumulation_steps=int(GRAD_ACCUM_STEPS),
    learning_rate=float(LEARNING_RATE),
    warmup_ratio=float(WARMUP_RATIO),
    warmup_steps=int(WARMUP_STEPS),
    min_lr_ratio=float(MIN_LR_RATIO),
    weight_decay=float(WEIGHT_DECAY),
    label_smoothing=float(LABEL_SMOOTHING),
    max_grad_norm=float(MAX_GRAD_NORM),
    num_workers=int(NUM_WORKERS),
    log_every_n_steps=int(LOG_EVERY_N_STEPS),
    save_every_n_steps=int(max(0, SAVE_EVERY_N_STEPS)),
    save_every_n_epochs=int(SAVE_EVERY_N_EPOCHS),
    keep_every_n_epochs=int(KEEP_EVERY_N_EPOCHS),
    max_checkpoints=int(MAX_CHECKPOINTS),
    auto_resume=bool(AUTO_RESUME),
    resume_from_checkpoint=str(RESUME_FROM_CHECKPOINT),
    resume_mode=str(RESUME_MODE),
    enable_data_parallel_c=False,
    enable_data_parallel_e=False,
    allow_gdn_data_parallel=False,
    allow_fallback_gdn=bool(ALLOW_FALLBACK_GDN),
    dry_run=bool(RUN_DRY_RUN),
)

use_ddp = bool(USE_DDP_FOR_MULTI_GPU and GPU_COUNT > 1 and REQUIRE_REAL_GDN)
estimated_steps_epoch = int(globals().get("ESTIMATED_STEPS_PER_EPOCH", 0))

if estimated_steps_epoch > 0:
    warmup_epoch_fraction = float(WARMUP_STEPS) / float(max(1, estimated_steps_epoch))
    print(
        f"Warmup configured to {int(WARMUP_STEPS):,} steps "
        f"(~{warmup_epoch_fraction:.2f} epoch, <= {WARMUP_MAX_EPOCH_FRACTION:.2f} epoch cap)."
    )

if int(SAVE_EVERY_N_STEPS) > 0:
    print(f"Step checkpoints configured every {int(SAVE_EVERY_N_STEPS):,} optimizer steps.")
else:
    print("Step checkpoints disabled (SAVE_EVERY_N_STEPS <= 0).")

print(f"Epoch export root: {EPOCH_EXPORT_ROOT}")
print(f"Epoch upload command configured: {bool(str(EPOCH_UPLOAD_CMD_TEMPLATE).strip())}")

# Scratch training is intentionally disabled in this notebook now.
if bool(RUN_TRAINING_FROM_SCRATCH):
    FINAL_RESULT_PATH = OUTPUT_DIR / "training_not_run.json"
    print("RUN_TRAINING_FROM_SCRATCH=True, but scratch launch is intentionally disabled.")
    print("Use resume flow (Cell 5 + RUN_RESUME_TRAINING=True) to continue from saved checkpoints.")
elif bool(RUN_DRY_RUN):
    dry_report = run_unified_sub100m(cfg)
    FINAL_RESULT_PATH = Path(dry_report["result_path"])
    print("Dry run complete.")
    print(f"Dry result path: {dry_report['result_path']}")
    print(f"Dry params: {dry_report['params']:,}")
elif bool(RUN_RESUME_TRAINING):
    if not str(RESUME_FROM_CHECKPOINT).strip():
        raise RuntimeError(
            "RUN_RESUME_TRAINING=True but RESUME_FROM_CHECKPOINT is empty. "
            "Run Cell 5 first or set RESUME_FROM_CHECKPOINT manually."
        )

    if use_ddp:
        ddp_script = PROJECT_ROOT / "training" / "train_variant_e_40m_ddp.py"
        if not ddp_script.exists():
            raise FileNotFoundError(f"Missing DDP trainer: {ddp_script}")

        ddp_batch_size = max(1, int(BATCH_PER_GPU))
        ddp_grad_accum = max(
            1,
            int(np.ceil(float(TARGET_EFFECTIVE_BATCH) / float(ddp_batch_size * int(GPU_COUNT)))),
        )

        torchrun_bin = shutil.which("torchrun")
        if torchrun_bin:
            launcher = [
                torchrun_bin,
                "--standalone",
                "--nnodes=1",
                "--nproc_per_node",
                str(int(GPU_COUNT)),
            ]
        else:
            launcher = [
                sys.executable,
                "-m",
                "torch.distributed.run",
                "--standalone",
                "--nnodes=1",
                "--nproc_per_node",
                str(int(GPU_COUNT)),
            ]

        cmd = [
            *launcher,
            str(ddp_script),
            "--pretokenized_manifest",
            str(PRETOKENIZED_MANIFEST),
            "--pretokenized_root",
            str(PRETOKENIZED_ROOT),
            "--output_dir",
            str(OUTPUT_DIR),
            "--max_pieces",
            str(int(MAX_PIECES)),
            "--seed",
            str(int(SEED)),
            "--seed_length",
            str(int(SEED_LENGTH)),
            "--continuation_length",
            str(int(CONTINUATION_LENGTH)),
            "--max_sequence_length",
            str(int(MAX_SEQUENCE_LENGTH)),
            "--epochs",
            str(int(EPOCHS)),
            "--batch_size",
            str(int(ddp_batch_size)),
            "--grad_accumulation_steps",
            str(int(ddp_grad_accum)),
            "--learning_rate",
            str(float(LEARNING_RATE)),
            "--warmup_ratio",
            str(float(WARMUP_RATIO)),
            "--warmup_steps",
            str(int(WARMUP_STEPS)),
            "--min_lr_ratio",
            str(float(MIN_LR_RATIO)),
            "--weight_decay",
            str(float(WEIGHT_DECAY)),
            "--label_smoothing",
            str(float(LABEL_SMOOTHING)),
            "--max_grad_norm",
            str(float(MAX_GRAD_NORM)),
            "--num_workers",
            str(int(NUM_WORKERS)),
            "--log_every_n_steps",
            str(int(LOG_EVERY_N_STEPS)),
            "--save_every_n_steps",
            str(int(max(0, SAVE_EVERY_N_STEPS))),
            "--save_every_n_epochs",
            str(int(SAVE_EVERY_N_EPOCHS)),
            "--resume_from_checkpoint",
            str(RESUME_FROM_CHECKPOINT),
            "--resume_mode",
            str(RESUME_MODE),
            "--epoch_bundle_root",
            str(EPOCH_EXPORT_ROOT),
        ]

        if bool(str(EPOCH_UPLOAD_CMD_TEMPLATE).strip()):
            cmd.extend(["--epoch_upload_cmd_template", str(EPOCH_UPLOAD_CMD_TEMPLATE).strip()])
        if bool(ALLOW_FALLBACK_GDN):
            cmd.append("--allow_fallback_gdn")

        env = os.environ.copy()
        env["PYTHONPATH"] = str(PROJECT_ROOT) + os.pathsep + env.get("PYTHONPATH", "")
        cpu_count = max(1, int(os.cpu_count() or 1))
        threads_per_rank = max(1, cpu_count // max(1, int(GPU_COUNT)))
        env.setdefault("OMP_NUM_THREADS", str(threads_per_rank))
        env.setdefault("MKL_NUM_THREADS", env["OMP_NUM_THREADS"] )

        print("Launching DDP resume run:")
        print(f"OMP_NUM_THREADS per rank: {env['OMP_NUM_THREADS']}")
        print(" ".join(cmd))
        subprocess.run(cmd, check=True, cwd=str(PROJECT_ROOT), env=env)

        FINAL_RESULT_PATH = OUTPUT_DIR / "variant_e_40m_ddp_result.json"
        print(f"DDP resume training complete: {FINAL_RESULT_PATH}")
    else:
        resumed_cfg = replace(
            cfg,
            dry_run=False,
            auto_resume=True,
            resume_from_checkpoint=str(RESUME_FROM_CHECKPOINT),
            resume_mode=str(RESUME_MODE),
        )
        final_report = run_unified_sub100m(resumed_cfg)
        FINAL_RESULT_PATH = Path(final_report["result_path"])
        print(f"Single-process resume training complete: {FINAL_RESULT_PATH}")
else:
    FINAL_RESULT_PATH = OUTPUT_DIR / "training_not_run.json"
    print("RUN_RESUME_TRAINING=False and RUN_DRY_RUN=False, so nothing was executed.")

if FINAL_RESULT_PATH.exists():
    payload = json.loads(FINAL_RESULT_PATH.read_text(encoding="utf-8"))
    print(f"Profile: {payload.get('profile')}")
    print(f"Backend: {payload.get('backend_status', {})}")
    print(f"Training profile: {payload.get('training_profile', {})}")

## Multi-GPU Note (Dual T4)

This notebook now prefers a DDP launch path when 2+ GPUs are available and real GDN is required.
- It avoids `DataParallel` for Variant E real-GDN runs.
- It uses one process per GPU via `torchrun`.
- It keeps a single-process fallback path for 1-GPU environments.

For Kaggle hardware choice:
- Prefer dual T4 with DDP for throughput.
- Use P100 only as a fallback when the environment cannot run real GDN kernels.

## Optional Next Architecture Candidate

If you want a stronger follow-up after this run, try a dual-rate GDN:
- Local stream for event-level detail (current Variant E style).
- Phrase stream that downsamples events and models long structure.
- Cross-gating from phrase stream into local stream updates.

This keeps the model mostly state-space/recurrent while improving long-range musical phrasing.

In [ ]:
from pathlib import Path
import importlib
import torch

from config import DataConfig
from generation.generate import GenerationConfig, generate_continuation
from utils.checkpoint_loading import (
    extract_data_config,
    load_model_from_checkpoint,
    load_tokenizer_for_checkpoint,
)
from utils.midi_utils import compare_pianorolls

display = None
Image = None
try:
    _ipd = importlib.import_module("IPython.display")
    display = getattr(_ipd, "display", None)
    Image = getattr(_ipd, "Image", None)
except Exception:
    pass

def _pick_generation_checkpoint() -> Path:
    roots = [
        OUTPUT_DIR / "checkpoints",
        Path("/kaggle/working/outputs/sub100m_e_40m_100k/checkpoints"),
        Path("/kaggle/input/models/chickaboomcmurtrie/variant-e-40m-gdn-piano/pytorch/default/1"),
        Path("/kaggle/input/variant-e-40m-gdn-piano"),
        Path("/kaggle/input/Variant-E-40M-GDN-Piano"),
    ]
    for filename in ("latest.safetensors", "best.safetensors"):
        candidates = []
        for root in roots:
            if root.exists():
                candidates.extend(root.rglob(filename))
        if candidates:
            candidates = sorted(candidates, key=lambda p: p.stat().st_mtime, reverse=True)
            return candidates[0]
    raise FileNotFoundError("No generation checkpoint (.safetensors) found in expected roots.")

def _pick_bluebird_seed() -> Path:
    candidates = [
        PROJECT_ROOT / "bluebird.mid",
        PROJECT_ROOT / "Bluebird.mid",
        Path("/kaggle/working/bluebird.mid"),
        Path("/kaggle/working/Bluebird.mid"),
    ]
    for path in candidates:
        if path.exists():
            return path
    matches = sorted(
        list(Path("/kaggle/working").rglob("bluebird.mid"))
        + list(Path("/kaggle/working").rglob("Bluebird.mid")),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    if matches:
        return matches[0]
    raise FileNotFoundError("Could not locate bluebird.mid in project root or /kaggle/working.")

checkpoint_path = _pick_generation_checkpoint()
seed_midi = _pick_bluebird_seed()
device = "cuda" if torch.cuda.is_available() else "cpu"

bundle = load_model_from_checkpoint(checkpoint_path, device=device, strict=True)
bundle.model.eval()

data_cfg_payload = extract_data_config(bundle.checkpoint_metadata)
data_cfg = DataConfig(**{
    key: value for key, value in data_cfg_payload.items() if key in DataConfig.__annotations__
})

tokenizer, tokenizer_meta = load_tokenizer_for_checkpoint(
    bundle.checkpoint_metadata,
    search_paths=[checkpoint_path.parent, OUTPUT_DIR, PROJECT_ROOT, Path("/kaggle/working")],
)

generation_dir = OUTPUT_DIR / "generated"
generation_dir.mkdir(parents=True, exist_ok=True)
output_midi = generation_dir / "bluebird_continuation.mid"
comparison_png = generation_dir / "bluebird_seed_vs_continuation.png"

gen_cfg = GenerationConfig(
    max_new_tokens=max(1, int(data_cfg.continuation_length)),
    temperature=0.9,
    top_p=0.95,
    top_k=50,
    repetition_penalty=1.1,
    repetition_window=64,
    min_tokens_to_keep=3,
    num_samples=1,
    max_consecutive_zero_deltas=12,
    save_continuation_only=True,
)

print(f"Seed MIDI: {seed_midi}")
print(f"Checkpoint: {checkpoint_path}")
print(f"Model class: {bundle.model_class}")
print(f"Tokenizer kind: {tokenizer_meta.get('tokenizer_kind')}")

generated_paths = generate_continuation(
    model=bundle.model,
    tokenizer=tokenizer,
    seed_midi_path=seed_midi,
    output_path=output_midi,
    config=data_cfg,
    generation_config=gen_cfg,
)

compare_pianorolls(seed_midi, generated_paths[0], save_path=comparison_png)
if callable(display) and Image is not None:
    display(Image(filename=str(comparison_png)))

print(f"Generated MIDI: {generated_paths[0]}")
print(f"Comparison PNG: {comparison_png}")